In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import folium
from folium.plugins import Fullscreen
from branca.element import Template, MacroElement
import geopandas as gpd
import fiona
from shapely.geometry import shape
import os
import json
import urllib.parse

out_dir = r"E:\Downloads"
input_mush = os.path.join(out_dir, "Mushroom Assessment.xlsx")
input_shp = os.path.join(out_dir, "NR2600 Cluster.shp")

print("1. Loading base Mushroom data...")
df_21 = pd.read_excel(input_mush, sheet_name='5G21')
df_26 = pd.read_excel(input_mush, sheet_name='5G26')

df_21.columns = [c.strip().lower() for c in df_21.columns]
df_26.columns = [c.strip().lower() for c in df_26.columns]

sites_21 = set(df_21['site_id'])
sites_26 = set(df_26['site_id'])

df_all = pd.concat([df_21, df_26]).drop_duplicates(subset=['site_id']).reset_index(drop=True)

def categorize(site_id):
    in_21 = site_id in sites_21
    in_26 = site_id in sites_26
    if in_21 and in_26: return 'NR21 & NR26'
    elif in_21: return 'NR21 Only'
    else: return 'NR26 Only'

df_all['Category'] = df_all['site_id'].apply(categorize)
df_all = df_all.dropna(subset=['lat', 'lon']).reset_index(drop=True)

print("2. Running Spatial Analysis (KNN)...")
df_all['lat_rad'] = np.radians(df_all['lat'])
df_all['lon_rad'] = np.radians(df_all['lon'])
coords = df_all[['lat_rad', 'lon_rad']].values
nbrs = NearestNeighbors(n_neighbors=6, metric='haversine').fit(coords)
distances, indices = nbrs.kneighbors(coords)

is_mushroom = []
for i in range(len(df_all)):
    if df_all.loc[i, 'Category'] != 'NR21 Only':
        is_mushroom.append(False)
        continue
    neighbor_idx = indices[i][1:]
    neighbor_cats = df_all.iloc[neighbor_idx]['Category'].values
    upgraded_count = sum(neighbor_cats == 'NR21 & NR26')
    is_mushroom.append(upgraded_count >= 3)

df_all['Is_Mushroom'] = is_mushroom
df_all.loc[df_all['Is_Mushroom'], 'Category'] = 'NR21 Mushroom'

# Professional Tech Palette
nr_colors = {
    'NR21 Only': '#3B82F6',       # Blue
    'NR26 Only': '#10B981',       # Emerald
    'NR21 & NR26': '#8B5CF6',     # Purple
    'NR21 Mushroom': '#EF4444'    # Red
}

print("3. Generating Map...")
records = []
try:
    with fiona.open(input_shp) as src:
        shp_crs = src.crs
        for f in src:
            records.append({'geometry': shape(f['geometry'])})
    gdf_border = gpd.GeoDataFrame(records, crs=shp_crs)
    if getattr(gdf_border, 'crs', None) is None:
        gdf_border.set_crs(epsg=4326, inplace=True)
    else:
        gdf_border = gdf_border.to_crs(epsg=4326)
except Exception as e:
    print(f"Warning: Could not load shapefile {{e}}")
    gdf_border = None

center_lat = df_all['lat'].mean()
center_lon = df_all['lon'].mean()
# Using dark_matter tiles
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB dark_matter', prefer_canvas=True)
Fullscreen(position='topleft', title='Full Screen', title_cancel='Exit Full Screen').add_to(m)

if gdf_border is not None:
    folium.GeoJson(
        gdf_border,
        name='Cluster Boundary',
        style_function=lambda x: {'fillColor': '#FFFFFF', 'color': '#FFFFFF', 'weight': 1, 'fillOpacity': 0.05, 'dashArray': '4, 4'}
    ).add_to(m)

for idx, row in df_all.iterrows():
    cat = row['Category']
    color = nr_colors.get(cat, 'gray')
    tooltip_html = f"<b>Site ID:</b> {row['site_id']}<br><b>Category:</b> {cat}"
    
    rad = 4
    if cat == 'NR21 Mushroom':
        rad = 8
        
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=rad,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        tooltip=tooltip_html
    ).add_to(m)

# Dark theme legend
legend_html = '''
{% macro html(this, kwargs) %}
<div style="position: fixed; 
     top: 15px; right: 15px; width: 200px; height: auto; 
     border:1px solid #374151; z-index:9999; font-size:14px;
     background-color: rgba(31, 41, 55, 0.95);
     padding: 16px; border-radius: 8px;
     font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; color: #F9FAFB;
     box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3);">
     <b style="font-size: 15px; margin-bottom: 12px; display: block; color: #F9FAFB;">Site Classification</b>
     <div style="margin-bottom: 8px;"><i style="background: #3B82F6; width: 14px; height: 14px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%;"></i> NR21 Only</div>
     <div style="margin-bottom: 8px;"><i style="background: #10B981; width: 14px; height: 14px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%;"></i> NR26 Only</div>
     <div style="margin-bottom: 8px;"><i style="background: #8B5CF6; width: 14px; height: 14px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%;"></i> NR21 & NR26</div>
     <div style="margin-bottom: 8px;"><i style="background: #EF4444; width: 14px; height: 14px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%;"></i> NR21 Mushroom</div>
</div>
{% endmacro %}
'''
macro = MacroElement()
macro._template = Template(legend_html)
m.get_root().add_child(macro)

# Render map to string and URI encode it safely
map_html_str = m.get_root().render()
encoded_map_html = urllib.parse.quote(map_html_str)

print("4. Generating Unified Master Dashboard...")
dashboard_html_path = os.path.join(out_dir, "NR_Mushroom_Master_Dashboard.html")
cat_counts = df_all['Category'].value_counts().to_dict()

df_mushrooms = df_all[df_all['Category'] == 'NR21 Mushroom'][['site_id', 'lat', 'lon']].copy()
df_mushrooms['lat'] = df_mushrooms['lat'].round(6)
df_mushrooms['lon'] = df_mushrooms['lon'].round(6)
table_data = df_mushrooms.to_dict(orient='records')

json_data = json.dumps(cat_counts)
color_map_json = json.dumps(nr_colors)
table_data_json = json.dumps(table_data)

total_sites = len(df_all)
nr26_infra = cat_counts.get('NR26 Only', 0) + cat_counts.get('NR21 & NR26', 0)
nr21_infra = cat_counts.get('NR21 Only', 0) + cat_counts.get('NR21 & NR26', 0) + cat_counts.get('NR21 Mushroom', 0)
mushroom_sites = cat_counts.get('NR21 Mushroom', 0)

html_template = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>NR Mushroom Master Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link href="https://cdn.jsdelivr.net/npm/gridjs/dist/theme/mermaid.min.css" rel="stylesheet" />
    <script src="https://cdn.jsdelivr.net/npm/gridjs/dist/gridjs.umd.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
    
    <style>
        :root {{
            --bg-color: #111827;
            --card-bg: #1F2937;
            --text-main: #F9FAFB;
            --text-muted: #9CA3AF;
            --border-color: #374151;
            --blue: #3B82F6;
            --green: #10B981;
            --purple: #8B5CF6;
            --red: #EF4444;
        }}
        body {{ 
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; 
            background-color: var(--bg-color); 
            color: var(--text-main); 
            margin: 0; 
            padding: 40px; 
        }}
        .container {{ max-width: 1200px; margin: 0 auto; }}
        .header {{ text-align: center; margin-bottom: 40px; }}
        .header h1 {{ margin: 0; font-size: 36px; font-weight: 800; color: var(--text-main); letter-spacing: -0.5px; }}
        .header p {{ color: var(--text-muted); font-size: 16px; margin-top: 8px; font-weight: 500; }}
        
        .kpi-container {{ display: flex; justify-content: center; gap: 24px; margin-bottom: 40px; flex-wrap: wrap; }}
        .kpi-card {{ 
            background: var(--card-bg); 
            border-radius: 16px; 
            padding: 24px; 
            width: 190px; 
            text-align: center; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2), 0 2px 4px -1px rgba(0, 0, 0, 0.1);
            border: 1px solid var(--border-color);
            transition: transform 0.2s ease, box-shadow 0.2s ease;
        }}
        .kpi-card:hover {{
            transform: translateY(-4px);
            box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.3), 0 4px 6px -2px rgba(0, 0, 0, 0.1);
        }}
        .kpi-title {{ font-size: 13px; color: var(--text-muted); text-transform: uppercase; margin-bottom: 12px; font-weight: 700; letter-spacing: 0.5px; }}
        .kpi-value {{ font-size: 42px; font-weight: 800; margin: 0; line-height: 1; }}
        
        .kpi-value.blue {{ color: var(--blue); }}
        .kpi-value.green {{ color: var(--green); }}
        .kpi-value.purple {{ color: var(--purple); }}
        .kpi-value.red {{ color: var(--red); }}
        
        .charts-container {{ display: flex; gap: 30px; justify-content: space-between; flex-wrap: wrap; margin-bottom: 40px; }}
        .chart-box {{ 
            background: var(--card-bg); 
            padding: 30px; 
            border-radius: 16px; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2); 
            border: 1px solid var(--border-color);
            flex: 1;
            min-width: 450px; 
            height: 350px;
        }}
        
        .table-container, .map-container {{
            background: var(--card-bg); 
            padding: 30px; 
            border-radius: 16px; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2); 
            border: 1px solid var(--border-color);
            margin-bottom: 40px;
        }}
        .table-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            border-bottom: 2px solid var(--bg-color);
            padding-bottom: 15px;
        }}
        .table-header h2 {{ margin: 0; font-size: 20px; color: var(--text-main); font-weight: 700; }}
        .table-badge {{ background: #451a1a; color: #EF4444; padding: 6px 14px; border-radius: 999px; font-size: 13px; font-weight: 700; border: 1px solid #7f1d1d; }}
        .table-badge.blue {{ background: #172554; color: #3B82F6; border-color: #1e3a8a; }}
        
        .export-btn {{
            background-color: var(--blue);
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 6px;
            font-family: 'Inter', sans-serif;
            font-weight: 600;
            font-size: 13px;
            cursor: pointer;
            transition: background-color 0.2s;
            display: flex;
            align-items: center;
            gap: 6px;
        }}
        .export-btn:hover {{ background-color: #2563EB; }}
        
        /* Grid.js Custom Overrides for Dark Mode */
        .gridjs-wrapper {{ border: none !important; box-shadow: none !important; border-radius: 8px !important; overflow: hidden; background: var(--card-bg) !important; }}
        .gridjs-th {{ background-color: #374151 !important; color: #F9FAFB !important; font-weight: 600 !important; font-size: 13px !important; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 1px solid #4B5563 !important; border-top: none !important; border-left: none !important; border-right: none !important; }}
        .gridjs-td {{ border: none !important; border-bottom: 1px solid #374151 !important; font-size: 14px; color: #F9FAFB !important; background: var(--card-bg) !important; }}
        .gridjs-search-input {{ background: #374151 !important; color: #F9FAFB !important; border-radius: 8px !important; border: 1px solid #4B5563 !important; padding: 10px 16px !important; width: 300px !important; font-family: 'Inter', sans-serif !important; outline: none; transition: border-color 0.2s; }}
        .gridjs-search-input:focus {{ border-color: var(--blue) !important; }}
        .gridjs-search-input::placeholder {{ color: #9CA3AF !important; }}
        .gridjs-pagination {{ background: var(--card-bg) !important; border-top: 1px solid #374151 !important; color: #9CA3AF !important; }}
        .gridjs-summary {{ color: #9CA3AF !important; }}
        .gridjs-pages button {{ background: #374151 !important; color: #F9FAFB !important; border: 1px solid #4B5563 !important; border-radius: 6px !important; font-family: 'Inter', sans-serif !important; margin: 0 2px !important; }}
        .gridjs-pages button:hover {{ background: #4B5563 !important; }}
        .gridjs-pages button[disabled] {{ opacity: 0.5 !important; cursor: not-allowed !important; }}
        .gridjs-container {{ color: var(--text-main) !important; }}
    </style>
</head>
<body>

<div class="container">
    <div class="header">
        <h1>NR Mushroom Analysis</h1>
        <p>Advanced Spatial Intelligence & Network Telemetry</p>
    </div>

    <div class="kpi-container">
        <div class="kpi-card">
            <div class="kpi-title">Total NR Sites</div>
            <div class="kpi-value" style="color: var(--text-main);">{total_sites}</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">NR26 Infrastructure</div>
            <div class="kpi-value green">{nr26_infra}</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">NR21 Infrastructure</div>
            <div class="kpi-value blue">{nr21_infra}</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">Critical Mushrooms</div>
            <div class="kpi-value red">{mushroom_sites}</div>
        </div>
    </div>

    <div class="charts-container">
        <div class="chart-box">
            <canvas id="barChart"></canvas>
        </div>
        <div class="chart-box" style="flex: 0.6; min-width: 350px;">
            <canvas id="pieChart"></canvas>
        </div>
    </div>

    <div class="table-container">
        <div class="table-header">
            <div style="display: flex; align-items: center; gap: 16px;">
                <h2>Critical Mushroom Sites Directory</h2>
                <span class="table-badge">{mushroom_sites} Actionable Sites</span>
            </div>
            <button onclick="exportToCSV()" class="export-btn">
                <svg width="16" height="16" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" d="M4 16v1a3 3 0 003 3h10a3 3 0 003-3v-1m-4-4l-4 4m0 0l-4-4m4 4V4"></path></svg>
                Export CSV
            </button>
        </div>
        <div id="wrapper"></div>
    </div>
    
    <div class="map-container">
        <div class="table-header">
            <div style="display: flex; align-items: center; gap: 16px;">
                <h2>Geospatial Network Visualization</h2>
                <span class="table-badge blue">Canvas Accelerated Rendering</span>
            </div>
        </div>
        <!-- Iframe container for the fully isolated Folium Map -->
        <iframe id="mapFrame" allowfullscreen="true" webkitallowfullscreen="true" mozallowfullscreen="true" style="width:100%; height:750px; border:none; border-radius:8px; background: #111827;"></iframe>
    </div>
</div>

<script>
    const data = {json_data};
    const colors = {color_map_json};
    const tableData = {table_data_json};
    
    // Safely decode the Folium Map HTML and inject it into the isolated iframe
    const mapHtmlString = decodeURIComponent("{encoded_map_html}");
    document.getElementById('mapFrame').srcdoc = mapHtmlString;
    
    // CSV Export Function
    function exportToCSV() {{
        const headers = ['Site ID', 'Latitude', 'Longitude'];
        const rows = tableData.map(r => [r.site_id, r.lat, r.lon]);
        
        let csvContent = "data:text/csv;charset=utf-8," 
            + headers.join(",") + "\\n" 
            + rows.map(e => e.join(",")).join("\\n");
            
        const encodedUri = encodeURI(csvContent);
        const link = document.createElement("a");
        link.setAttribute("href", encodedUri);
        link.setAttribute("download", "Mushroom_Sites_Export.csv");
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }}

    const labels = ['NR21 Only', 'NR26 Only', 'NR21 & NR26', 'NR21 Mushroom'];
    const values = labels.map(l => data[l] || 0);
    const bgColors = labels.map(l => colors[l]);

    Chart.defaults.color = '#9CA3AF';
    Chart.defaults.font.family = "'Inter', sans-serif";

    // Bar Chart
    new Chart(document.getElementById('barChart'), {{
        type: 'bar',
        data: {{
            labels: ['NR21 Only', 'NR26 Only', 'NR21 & NR26', 'Mushroom'],
            datasets: [{{
                label: 'Site Count',
                data: values,
                backgroundColor: bgColors,
                borderRadius: 6,
                borderSkipped: false
            }}]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{ 
                legend: {{ display: false }}, 
                title: {{ display: true, text: 'Infrastructure Distribution', font: {{ size: 16, weight: 600 }}, color: '#F9FAFB', padding: {{ bottom: 20 }}, align: 'start' }} 
            }},
            scales: {{ 
                y: {{ beginAtZero: true, grid: {{ color: '#374151' }}, border: {{ display: false }} }}, 
                x: {{ grid: {{ display: false }}, border: {{ display: false }} }} 
            }}
        }}
    }});

    // Doughnut Chart
    new Chart(document.getElementById('pieChart'), {{
        type: 'doughnut',
        data: {{
            labels: ['NR21 Only', 'NR26 Only', 'NR21 & NR26', 'Mushroom'],
            datasets: [{{
                data: values,
                backgroundColor: bgColors,
                borderWidth: 0,
                hoverOffset: 8
            }}]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false,
            cutout: '65%',
            plugins: {{ 
                legend: {{ position: 'right', labels: {{ padding: 20, usePointStyle: true, pointStyle: 'circle', color: '#9CA3AF' }} }},
                title: {{ display: true, text: 'Site Composition', font: {{ size: 16, weight: 600 }}, color: '#F9FAFB', padding: {{ bottom: 20 }}, align: 'start' }}
            }}
        }}
    }});

    // Render Grid.js Data Table
    new gridjs.Grid({{
        columns: [
            {{ id: 'site_id', name: 'Site ID' }},
            {{ id: 'lat', name: 'Latitude' }},
            {{ id: 'lon', name: 'Longitude' }}
        ],
        data: tableData,
        search: true,
        sort: true,
        pagination: {{
            enabled: true,
            limit: 10
        }},
        language: {{
            search: {{ placeholder: 'Search sites...' }}
        }}
    }}).render(document.getElementById('wrapper'));
</script>

</body>
</html>
"""

with open(dashboard_html_path, "w", encoding="utf-8") as f:
    f.write(html_template)
print(f"  -> Exported: {dashboard_html_path}")

print("5. Exporting Summary...")
excel_out = os.path.join(out_dir, "NR_Mushroom_Summary.xlsx")
df_all[['site_id', 'site_name', 'lon', 'lat', 'Category']].to_excel(excel_out, index=False)
print(f"  -> Exported: {excel_out}")
